In [1]:
from pathlib import Path
import importlib
import sys

import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

import dataloader
import audio_feature_dataloader
import processed_feature_builder
importlib.reload(dataloader)
importlib.reload(audio_feature_dataloader)
importlib.reload(processed_feature_builder)

from dataloader import load_data
from audio_feature_dataloader import build_audio_feature_dataset
from processed_feature_builder import build_processed_two_tower_data, save_processed_two_tower_data

DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SENSOR_CSV = DATA_DIR / "wp_node_gps.csv"
GPS_CSV = DATA_DIR / "20260416_150152_gps2_gps.csv"

SAMPLE_MS = 200
RESAMPLE_RATE = f"{SAMPLE_MS}ms"
CHUNK_MS = 10
EXPANDED_PREFIX = "vehicle_sensor_subset_200ms_expanded_features"

context_audio_features = [
    "rms_db", "spectral_centroid_hz", "spectral_flatness",
    "band_20_120_ratio", "band_120_500_ratio", "band_500_2000_ratio",
    "low_to_voice_db", "mid_to_voice_db", "rms_delta_db", "centroid_delta_hz",
    "peak_db", "crest_factor_db", "zcr", "spectral_bandwidth_hz",
    "spectral_rolloff85_hz", "spectral_entropy", "band_20_120_db",
    "band_120_500_db", "band_500_2000_db", "band_2000_6000_db",
    "band_2000_6000_ratio", "flatness_delta",
]

action_audio_features = [
    "rms_db", "spectral_centroid_hz", "spectral_flatness",
    "band_20_120_ratio", "band_120_500_ratio", "low_to_voice_db",
    "band_500_2000_ratio", "mid_to_voice_db", "crest_factor_db", "zcr",
    "spectral_bandwidth_hz", "spectral_rolloff85_hz", "spectral_entropy",
    "band_2000_6000_ratio", "flatness_delta",
]

per_node_required = [
    "rssi_median", "rssi_mad", "rssi_iqr", "rssi_trimmed_mean",
    "rssi_persistence_top3", "rssi_persistence_top5",
    "vehicle_like_score", "construction_like_score", "vehicle_minus_construction_score",
]
global_required = [
    "rssi_top1_top3_gap", "rssi_top1_top5_gap", "global_rssi_mean",
    "global_rssi_std", "global_rssi_range",
] + [
    f"global_{stat}_{feature}"
    for feature in [
        "construction_like_score", "vehicle_like_score", "spectral_entropy",
        "crest_factor_db", "band_2000_6000_ratio",
    ]
    for stat in ["mean", "max", "std", "range"]
]
action_subset_required = [
    "subset_centroid_minus_acoustic_com_x",
    "subset_centroid_minus_acoustic_com_y",
    "subset_centroid_to_acoustic_com_dist",
    "num_signal_top2_in_subset", "num_signal_top3_in_subset", "num_signal_top5_in_subset",
    "best_signal_rank_in_subset", "mean_signal_rank_in_subset", "worst_signal_rank_in_subset",
    "median_rssi", "mad_rssi", "iqr_rssi", "range_rssi",
]
audio_agg_required = [
    f"{stat}_audio_{feature}"
    for feature in action_audio_features
    for stat in ["mean", "max", "min", "std", "range"]
]
score_agg_required = [
    f"{stat}_{feature}"
    for feature in ["construction_like_score", "vehicle_like_score", "vehicle_minus_construction_score"]
    for stat in ["mean", "max", "std"]
]

node_list = sorted(pd.read_csv(SENSOR_CSV)["Node #"].astype(int).tolist())
gdf_cleaned, normalized_cleaned, valid_indices, gdf_nodes = load_data(
    data_dir=DATA_DIR,
    sensor_csv=SENSOR_CSV,
    gps_csv=GPS_CSV,
    node_list=node_list,
    chunk_ms=CHUNK_MS,
    sample_ms=SAMPLE_MS,
    resample_rate=RESAMPLE_RATE,
    plot_gaps=False,
    interpolate_gps=True,
)
audio_feature_df, audio_feature_long, audio_gdf_nodes = build_audio_feature_dataset(
    data_dir=DATA_DIR,
    sensor_csv=SENSOR_CSV,
    node_list=node_list,
    sample_ms=SAMPLE_MS,
    add_cross_sensor=True,
)

expanded_processed = build_processed_two_tower_data(
    gdf_cleaned=gdf_cleaned,
    gdf_nodes=gdf_nodes,
    valid_indices=valid_indices,
    node_list=node_list,
    audio_feature_long=audio_feature_long,
    history_steps=5,
    max_subset_size=3,
    utility_second_weight=0.45,
    utility_third_weight=0.20,
    softmax_temperature=4.0,
    context_audio_features=context_audio_features,
    action_audio_features=action_audio_features,
    include_audio_derived_features=True,
    verbose=True,
    progress_every=1500,
)
expanded_paths = save_processed_two_tower_data(
    expanded_processed,
    processed_dir=PROCESSED_DIR,
    prefix=EXPANDED_PREFIX,
)

meta = expanded_processed["meta"]
missing_context_audio = sorted(set(context_audio_features) - set(meta["context_audio_features"]))
missing_action_audio = sorted(set(action_audio_features) - set(meta["action_audio_features"]))
assert not missing_context_audio, missing_context_audio
assert not missing_action_audio, missing_action_audio

context_names = set(meta["context_feature_names"])
node_feature_cols = set(expanded_processed["node_feature_df"].columns)
feature_wide_cols = set(expanded_processed["feature_wide_df"].columns)
for node in meta["ordered_nodes"]:
    for feature in per_node_required:
        assert feature in node_feature_cols, f"missing node_feature_df column: {feature}"
        assert f"n{node}_{feature}" in context_names, f"missing context feature: n{node}_{feature}"
        assert f"n{node}_{feature}" in feature_wide_cols, f"missing feature_wide column: n{node}_{feature}"
    for feature in context_audio_features:
        assert f"n{node}_audio_{feature}" in context_names, f"missing context audio: n{node}_audio_{feature}"

global_names = set(meta["global_feature_names"])
for feature in global_required:
    assert feature in global_names, f"missing global context feature: {feature}"
    assert feature in feature_wide_cols, f"missing feature_wide global column: {feature}"

action_names = set(meta["action_feature_names"])
for feature in action_subset_required + audio_agg_required + score_agg_required:
    assert feature in action_names, f"missing action feature: {feature}"

arrays = np.load(expanded_paths["arrays_npz"])
assert arrays["C_by_time"].shape[1] == meta["context_dim"] == len(meta["context_feature_names"])
assert arrays["A_examples"].shape[1] == meta["action_raw_dim"] == len(meta["action_feature_names"])

leak_tokens = ("distance_to", "vehicle_x", "vehicle_y", "latitude", "longitude")
feature_names = list(meta["context_feature_names"]) + list(meta["action_feature_names"])
leaky_features = [name for name in feature_names if any(tok in name.lower() for tok in leak_tokens)]
assert not leaky_features, leaky_features[:20]

print("expanded processed prefix:", EXPANDED_PREFIX)
print("context_dim:", meta["context_dim"])
print("action_dim:", meta["action_raw_dim"])
print("times/examples:", meta["num_times"], meta["num_examples"])
print("all requested expanded features are present; no feature-side vehicle coordinate leakage detected")
display(pd.DataFrame([{"artifact": k, "path": str(v)} for k, v in expanded_paths.items()]))


Loaded FLAC: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26- MILCOM Attempt\data\raw\20260416_150152_dvpg_gq_orin_11_respeaker.flac
Loaded FLAC: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26- MILCOM Attempt\data\raw\20260416_150152_dvpg_gq_orin_12_respeaker.flac
Loaded FLAC: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26- MILCOM Attempt\data\raw\20260416_150152_dvpg_gq_orin_13_respeaker.flac
Loaded FLAC: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26- MILCOM Attempt\data\raw\20260416_150152_dvpg_gq_orin_14_respeaker.flac
Loaded FLAC: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26- MILCOM Attempt\data\raw\20260416_150152_dvpg_gq_orin_15_respeaker.flac
Loaded FLAC: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26- MILCOM Attempt\data\raw\20260416_150152_dvpg_gq_orin_16_respeaker.flac
Node 11: 155480 rows loaded, time range: 2026-04-16 15:01:52 to 2026-04-16 15:27:46.790000
Node 12: 155490 rows loaded, time range: 2026-04-16 15:01:52 to 2026-04-16 15:27:46.890000
Node 13: 15549

,artifact,path
0,feature_wide_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
1,node_features_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
2,ground_truth_nodes_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
3,ground_truth_vehicle_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
4,sensor_geometry_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
5,sequence_pkl,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
6,examples_pkl,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
7,examples_index_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
8,arrays_npz,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
9,meta_json,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...


In [2]:
import importlib
import two_tower_training
importlib.reload(two_tower_training)

from two_tower_training import TrainConfig, compare_on_common_objectives, run_config_grid, summarize_results

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVED_OBJECTIVES = [
    {"name": "saved_rational", "utility_name": "saved", "utility_kwargs": {}},
    {"name": "contains_closest", "utility_name": "closest_binary", "utility_kwargs": {}},
]

expanded_h512_config = TrainConfig(
    run_name="expanded_saved_h512_d2_e8",
    utility_name="saved",
    hidden=512,
    emb_dim=8,
    depth=2,
    dropout=0.05,
    combine_mode="mul_only",
    loss_name="mse",
    max_epochs=50,
    patience=16,
    batch_size=8192,
    lr=1e-3,
    weight_decay=1e-4,
    seed=22,
    log_every=1,
)

expanded_results = run_config_grid(
    PROCESSED_DIR,
    [expanded_h512_config],
    prefix=EXPANDED_PREFIX,
    device=DEVICE,
)
expanded_summary_df = summarize_results(expanded_results)
expanded_common_eval_df = compare_on_common_objectives(expanded_results, SAVED_OBJECTIVES)
display(expanded_summary_df)
display(expanded_common_eval_df[expanded_common_eval_df["split"].eq("test")].reset_index(drop=True))


expanded_saved_h512_d2_e8 ep 001/050* loss=0.04510 val_rmse=0.1434 top1=0.238 top3=0.449 reg=0.0357 rank=5.98 11s
expanded_saved_h512_d2_e8 ep 002/050  loss=0.01762 val_rmse=0.1354 top1=0.147 top3=0.391 reg=0.0406 rank=6.46 22s
expanded_saved_h512_d2_e8 ep 003/050  loss=0.01254 val_rmse=0.1297 top1=0.162 top3=0.385 reg=0.0362 rank=6.03 33s
expanded_saved_h512_d2_e8 ep 004/050* loss=0.00994 val_rmse=0.1244 top1=0.161 top3=0.419 reg=0.0315 rank=5.53 44s
expanded_saved_h512_d2_e8 ep 005/050* loss=0.00827 val_rmse=0.1247 top1=0.185 top3=0.453 reg=0.0289 rank=5.23 55s
expanded_saved_h512_d2_e8 ep 006/050* loss=0.00705 val_rmse=0.1262 top1=0.203 top3=0.466 reg=0.0287 rank=5.20 66s
expanded_saved_h512_d2_e8 ep 007/050* loss=0.00613 val_rmse=0.1244 top1=0.213 top3=0.487 reg=0.0281 rank=5.14 77s
expanded_saved_h512_d2_e8 ep 008/050* loss=0.00544 val_rmse=0.1261 top1=0.243 top3=0.500 reg=0.0274 rank=5.05 90s
expanded_saved_h512_d2_e8 ep 009/050* loss=0.00490 val_rmse=0.1221 top1=0.238 top3=0.508

,run_name,utility,hidden,emb_dim,depth,dropout,combine,loss,best_epoch,train_rmse,...,val_avg_regret,val_avg_norm_regret,test_rmse,test_mae,test_r2,test_top1,test_top3,test_mean_rank,test_avg_regret,test_avg_norm_regret
0,expanded_saved_h512_d2_e8,saved,512,8,2,0.05,mul_only,mse,41,0.019546,...,0.022904,0.050454,0.116675,0.083843,0.656293,0.283322,0.544541,4.425318,0.023707,0.047672


,run_name,train_utility,eval_objective,split,hidden,emb_dim,combine,top1,top3,mean_rank,avg_regret,avg_norm_regret
0,expanded_saved_h512_d2_e8,saved,contains_closest,test,512,8,mul_only,0.943068,0.943068,1.910918,0.056932,0.056932
1,expanded_saved_h512_d2_e8,saved,saved_rational,test,512,8,mul_only,0.283322,0.544541,4.425318,0.023707,0.047672


In [4]:
# Same model and saved utility, but no waveform/frequency features beyond RSSI-derived features.
# Sensor GPS geometry and RSSI history/rank/persistence features remain available.

RSSI_GEOMETRY_PREFIX = "vehicle_sensor_subset_200ms_rssi_geometry_only"

rssi_geometry_processed = build_processed_two_tower_data(
    gdf_cleaned=gdf_cleaned,
    gdf_nodes=gdf_nodes,
    valid_indices=valid_indices,
    node_list=node_list,
    audio_feature_long=None,
    history_steps=5,
    max_subset_size=3,
    utility_second_weight=0.45,
    utility_third_weight=0.20,
    softmax_temperature=4.0,
    context_audio_features=[],
    action_audio_features=[],
    include_audio_derived_features=False,
    verbose=True,
    progress_every=1500,
)
rssi_geometry_paths = save_processed_two_tower_data(
    rssi_geometry_processed,
    processed_dir=PROCESSED_DIR,
    prefix=RSSI_GEOMETRY_PREFIX,
)

rssi_meta = rssi_geometry_processed["meta"]
assert rssi_meta["context_audio_features"] == []
assert rssi_meta["action_audio_features"] == []
assert rssi_meta["derived_audio_score_features"] == []
assert not [name for name in rssi_meta["context_feature_names"] if "audio_" in name]
assert not [name for name in rssi_meta["action_feature_names"] if "audio_" in name]

rssi_geometry_h512_config = TrainConfig(
    run_name="rssi_geometry_saved_h512_d2_e8",
    utility_name="saved",
    hidden=512,
    emb_dim=8,
    depth=2,
    dropout=0.05,
    combine_mode="mul_only",
    loss_name="mse",
    max_epochs=50,
    patience=16,
    batch_size=8192,
    lr=1e-3,
    weight_decay=1e-4,
    seed=22,
    log_every=1,
)

rssi_geometry_results = run_config_grid(
    PROCESSED_DIR,
    [rssi_geometry_h512_config],
    prefix=RSSI_GEOMETRY_PREFIX,
    device=DEVICE,
)
rssi_geometry_summary_df = summarize_results(rssi_geometry_results)
rssi_geometry_common_eval_df = compare_on_common_objectives(rssi_geometry_results, SAVED_OBJECTIVES)

display(rssi_geometry_summary_df)
display(rssi_geometry_common_eval_df[rssi_geometry_common_eval_df["split"].eq("test")].reset_index(drop=True))

if "expanded_results" in globals():
    ablation_eval_df = compare_on_common_objectives(
        expanded_results + rssi_geometry_results,
        SAVED_OBJECTIVES,
    )
    display(
        ablation_eval_df[ablation_eval_df["split"].eq("test")]
        .sort_values(["eval_objective", "avg_norm_regret", "mean_rank"])
        .reset_index(drop=True)
    )


valid timesteps: 7465
ordered nodes: [11, 12, 13, 14, 15, 16]
temporal features: 1/7465
temporal features: 1500/7465
temporal features: 3000/7465
temporal features: 4500/7465
temporal features: 6000/7465
temporal features: 7465/7465
node rows: 1/7465
node rows: 1500/7465
node rows: 3000/7465
node rows: 4500/7465
node rows: 6000/7465
node rows: 7465/7465
action rows: 1/7465
action rows: 1500/7465
action rows: 3000/7465
action rows: 4500/7465
action rows: 6000/7465
action rows: 7465/7465
processed two-tower data ready
sequence_df: (7465, 5)
examples_df: (306065, 15)
context_dim: 99
action_raw_dim: 76
rssi_geometry_saved_h512_d2_e8 ep 001/050* loss=0.04696 val_rmse=0.1697 top1=0.204 top3=0.352 reg=0.0712 rank=8.17 11s
rssi_geometry_saved_h512_d2_e8 ep 002/050* loss=0.02651 val_rmse=0.1692 top1=0.173 top3=0.293 reg=0.0678 rank=8.62 22s
rssi_geometry_saved_h512_d2_e8 ep 003/050  loss=0.02266 val_rmse=0.1787 top1=0.175 top3=0.281 reg=0.0820 rank=9.99 33s
rssi_geometry_saved_h512_d2_e8 ep 004

,run_name,utility,hidden,emb_dim,depth,dropout,combine,loss,best_epoch,train_rmse,...,val_avg_regret,val_avg_norm_regret,test_rmse,test_mae,test_r2,test_top1,test_top3,test_mean_rank,test_avg_regret,test_avg_norm_regret
0,rssi_geometry_saved_h512_d2_e8,saved,512,8,2,0.05,mul_only,mse,2,0.149376,...,0.067825,0.135486,0.160671,0.123608,0.348208,0.22639,0.330208,8.073677,0.072984,0.134118


,run_name,train_utility,eval_objective,split,hidden,emb_dim,combine,top1,top3,mean_rank,avg_regret,avg_norm_regret
0,rssi_geometry_saved_h512_d2_e8,saved,contains_closest,test,512,8,mul_only,0.77361,0.773610,4.622237,0.226390,0.226390
1,rssi_geometry_saved_h512_d2_e8,saved,saved_rational,test,512,8,mul_only,0.22639,0.330208,8.073677,0.072984,0.134118


,run_name,train_utility,eval_objective,split,hidden,emb_dim,combine,top1,top3,mean_rank,avg_regret,avg_norm_regret
0,expanded_saved_h512_d2_e8,saved,contains_closest,test,512,8,mul_only,0.943068,0.943068,1.910918,0.056932,0.056932
1,rssi_geometry_saved_h512_d2_e8,saved,contains_closest,test,512,8,mul_only,0.773610,0.773610,4.622237,0.226390,0.226390
2,expanded_saved_h512_d2_e8,saved,saved_rational,test,512,8,mul_only,0.283322,0.544541,4.425318,0.023707,0.047672
3,rssi_geometry_saved_h512_d2_e8,saved,saved_rational,test,512,8,mul_only,0.226390,0.330208,8.073677,0.072984,0.134118


In [5]:
import importlib
import pandas as pd
from dataclasses import replace

import two_tower_training
importlib.reload(two_tower_training)

from two_tower_training import (
    TrainConfig,
    compare_on_common_objectives,
    run_config_grid,
    summarize_results,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SAVED_OBJECTIVES = [
    {"name": "saved_rational", "utility_name": "saved", "utility_kwargs": {}},
    {"name": "contains_closest", "utility_name": "closest_binary", "utility_kwargs": {}},
]

# Your current expanded baseline.
base_config = TrainConfig(
    run_name="expanded_saved_h512_d2_e8",
    utility_name="saved",
    hidden=512,
    emb_dim=8,
    depth=2,
    dropout=0.05,
    combine_mode="mul_only",
    loss_name="mse",
    max_epochs=50,
    patience=16,
    batch_size=8192,
    lr=1e-3,
    weight_decay=1e-4,
    seed=22,
    log_every=1,
)

# One-change-at-a-time configs around the baseline.
one_change_configs = [
    # 1. Main bottleneck test: 8 -> 16 embedding.
    replace(
        base_config,
        run_name="expanded_saved_h512_d2_e16",
        emb_dim=16,
    ),

    # 2. Stronger bottleneck test: 8 -> 24 embedding.
    # Still not insane for the later linearized bandit: d = 1 + 3*24 = 73.
    replace(
        base_config,
        run_name="expanded_saved_h512_d2_e24",
        emb_dim=24,
    ),

    # 3. Dropout regularization test.
    replace(
        base_config,
        run_name="expanded_saved_h512_d2_e8_drop010",
        dropout=0.10,
    ),

    # 4. Weight-decay regularization test.
    replace(
        base_config,
        run_name="expanded_saved_h512_d2_e8_wd5e4",
        weight_decay=5e-4,
    ),

    # 5. Noisy-label robustness test.
    replace(
        base_config,
        run_name="expanded_saved_h512_d2_e8_smoothl1",
        loss_name="smooth_l1",
    ),
]

one_change_results = run_config_grid(
    PROCESSED_DIR,
    one_change_configs,
    prefix=EXPANDED_PREFIX,
    device=DEVICE,
)

one_change_summary_df = summarize_results(one_change_results)

one_change_common_eval_df = compare_on_common_objectives(
    one_change_results,
    SAVED_OBJECTIVES,
)

display(one_change_summary_df)

display(
    one_change_common_eval_df[
        one_change_common_eval_df["split"].eq("test")
    ].reset_index(drop=True)
)

expanded_saved_h512_d2_e16 ep 001/050* loss=0.04178 val_rmse=0.1456 top1=0.217 top3=0.395 reg=0.0408 rank=6.45 12s
expanded_saved_h512_d2_e16 ep 002/050* loss=0.01627 val_rmse=0.1311 top1=0.172 top3=0.376 reg=0.0354 rank=6.07 23s
expanded_saved_h512_d2_e16 ep 003/050* loss=0.01144 val_rmse=0.1313 top1=0.168 top3=0.376 reg=0.0352 rank=6.04 35s
expanded_saved_h512_d2_e16 ep 004/050* loss=0.00891 val_rmse=0.1335 top1=0.200 top3=0.405 reg=0.0339 rank=5.81 47s
expanded_saved_h512_d2_e16 ep 005/050* loss=0.00730 val_rmse=0.1322 top1=0.202 top3=0.420 reg=0.0325 rank=5.67 59s
expanded_saved_h512_d2_e16 ep 006/050* loss=0.00617 val_rmse=0.1306 top1=0.223 top3=0.440 reg=0.0298 rank=5.37 71s
expanded_saved_h512_d2_e16 ep 007/050* loss=0.00532 val_rmse=0.1275 top1=0.228 top3=0.433 reg=0.0296 rank=5.34 83s
expanded_saved_h512_d2_e16 ep 008/050* loss=0.00470 val_rmse=0.1253 top1=0.228 top3=0.453 reg=0.0273 rank=5.09 95s
expanded_saved_h512_d2_e16 ep 009/050* loss=0.00422 val_rmse=0.1266 top1=0.262 t

,run_name,utility,hidden,emb_dim,depth,dropout,combine,loss,best_epoch,train_rmse,...,val_avg_regret,val_avg_norm_regret,test_rmse,test_mae,test_r2,test_top1,test_top3,test_mean_rank,test_avg_regret,test_avg_norm_regret
0,expanded_saved_h512_d2_e16,saved,512,16,2,0.05,mul_only,mse,49,0.015718,...,0.021587,0.047695,0.117341,0.084514,0.652360,0.358339,0.606162,4.034159,0.021643,0.042809
1,expanded_saved_h512_d2_e8_smoothl1,saved,512,8,2,0.05,mul_only,smooth_l1,8,0.045941,...,0.022192,0.048724,0.119032,0.088301,0.642266,0.215673,0.482920,4.508372,0.023325,0.046852
2,expanded_saved_h512_d2_e8_drop010,saved,512,8,2,0.10,mul_only,mse,30,0.029872,...,0.022825,0.050606,0.118083,0.085402,0.647949,0.279303,0.535164,4.306095,0.023078,0.045778
3,expanded_saved_h512_d2_e8_wd5e4,saved,512,8,2,0.05,mul_only,mse,41,0.019413,...,0.022854,0.050628,0.115975,0.083389,0.660402,0.282652,0.549230,4.342264,0.023081,0.046444
4,expanded_saved_h512_d2_e24,saved,512,24,2,0.05,mul_only,mse,28,0.024481,...,0.023883,0.049692,0.123081,0.090066,0.617515,0.249163,0.459478,4.718687,0.025646,0.050379


,run_name,train_utility,eval_objective,split,hidden,emb_dim,combine,top1,top3,mean_rank,avg_regret,avg_norm_regret
0,expanded_saved_h512_d2_e8_smoothl1,saved,contains_closest,test,512,8,mul_only,0.957803,0.957803,1.675151,0.042197,0.042197
1,expanded_saved_h512_d2_e8_drop010,saved,contains_closest,test,512,8,mul_only,0.950435,0.950435,1.793034,0.049565,0.049565
2,expanded_saved_h512_d2_e8_wd5e4,saved,contains_closest,test,512,8,mul_only,0.945747,0.945747,1.868051,0.054253,0.054253
3,expanded_saved_h512_d2_e16,saved,contains_closest,test,512,16,mul_only,0.945077,0.945077,1.878768,0.054923,0.054923
4,expanded_saved_h512_d2_e24,saved,contains_closest,test,512,24,mul_only,0.942398,0.942398,1.921634,0.057602,0.057602
5,expanded_saved_h512_d2_e16,saved,saved_rational,test,512,16,mul_only,0.358339,0.606162,4.034159,0.021643,0.042809
6,expanded_saved_h512_d2_e8_drop010,saved,saved_rational,test,512,8,mul_only,0.279303,0.535164,4.306095,0.023078,0.045778
7,expanded_saved_h512_d2_e8_wd5e4,saved,saved_rational,test,512,8,mul_only,0.282652,0.549230,4.342264,0.023081,0.046444
8,expanded_saved_h512_d2_e8_smoothl1,saved,saved_rational,test,512,8,mul_only,0.215673,0.482920,4.508372,0.023325,0.046852
9,expanded_saved_h512_d2_e24,saved,saved_rational,test,512,24,mul_only,0.249163,0.459478,4.718687,0.025646,0.050379


In [7]:
import importlib
import pandas as pd
from dataclasses import replace

import two_tower_training
importlib.reload(two_tower_training)

from two_tower_training import (
    TrainConfig,
    compare_on_common_objectives,
    run_config_grid,
    summarize_results,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SAVED_OBJECTIVES = [
    {"name": "saved_rational", "utility_name": "saved", "utility_kwargs": {}},
    {"name": "contains_closest", "utility_name": "closest_binary", "utility_kwargs": {}},
]

# New baseline: best from the previous batch.
base_e16_config = TrainConfig(
    run_name="expanded_saved_h512_d2_e16",
    utility_name="saved",
    hidden=512,
    emb_dim=16,
    depth=2,
    dropout=0.05,
    combine_mode="mul_only",
    loss_name="mse",
    max_epochs=50,
    patience=16,
    batch_size=8192,
    lr=1e-3,
    weight_decay=1e-4,
    seed=22,
    log_every=1,
)

e16_probe_configs = [
    # 1. Smaller embedding near the winner.
    replace(
        base_e16_config,
        run_name="expanded_saved_h512_d2_e12",
        emb_dim=12,
    ),

    # 2. Larger embedding near the winner, but below the bad e24 case.
    replace(
        base_e16_config,
        run_name="expanded_saved_h512_d2_e20",
        emb_dim=20,
    ),

    # 3. More dropout around e16.
    replace(
        base_e16_config,
        run_name="expanded_saved_h512_d2_e16_drop010",
        dropout=0.10,
    ),

    # 4. More weight decay around e16.
    replace(
        base_e16_config,
        run_name="expanded_saved_h512_d2_e16_wd5e4",
        weight_decay=5e-4,
    ),

    # 5. Lower learning rate around e16.
    replace(
        base_e16_config,
        run_name="expanded_saved_h512_d2_e16_lr5e4",
        lr=5e-4,
    ),
]

e16_probe_results = run_config_grid(
    PROCESSED_DIR,
    e16_probe_configs,
    prefix=EXPANDED_PREFIX,
    device=DEVICE,
)

e16_probe_summary_df = summarize_results(e16_probe_results)

e16_probe_common_eval_df = compare_on_common_objectives(
    e16_probe_results,
    SAVED_OBJECTIVES,
)

display(e16_probe_summary_df)

display(
    e16_probe_common_eval_df[
        e16_probe_common_eval_df["split"].eq("test")
    ].reset_index(drop=True)
)

expanded_saved_h512_d2_e12 ep 001/050* loss=0.05611 val_rmse=0.1463 top1=0.152 top3=0.393 reg=0.0432 rank=6.77 13s
expanded_saved_h512_d2_e12 ep 002/050* loss=0.01861 val_rmse=0.1312 top1=0.126 top3=0.354 reg=0.0378 rank=6.30 23s
expanded_saved_h512_d2_e12 ep 003/050* loss=0.01319 val_rmse=0.1313 top1=0.123 top3=0.386 reg=0.0348 rank=6.06 34s
expanded_saved_h512_d2_e12 ep 004/050* loss=0.01049 val_rmse=0.1302 top1=0.167 top3=0.420 reg=0.0306 rank=5.56 44s
expanded_saved_h512_d2_e12 ep 005/050* loss=0.00875 val_rmse=0.1312 top1=0.210 top3=0.475 reg=0.0278 rank=5.17 55s
expanded_saved_h512_d2_e12 ep 006/050* loss=0.00746 val_rmse=0.1302 top1=0.217 top3=0.497 reg=0.0260 rank=4.93 65s
expanded_saved_h512_d2_e12 ep 007/050* loss=0.00656 val_rmse=0.1310 top1=0.220 top3=0.491 reg=0.0255 rank=4.89 75s
expanded_saved_h512_d2_e12 ep 008/050  loss=0.00583 val_rmse=0.1304 top1=0.222 top3=0.490 reg=0.0258 rank=4.94 85s
expanded_saved_h512_d2_e12 ep 009/050* loss=0.00520 val_rmse=0.1314 top1=0.236 t

,run_name,utility,hidden,emb_dim,depth,dropout,combine,loss,best_epoch,train_rmse,...,val_avg_regret,val_avg_norm_regret,test_rmse,test_mae,test_r2,test_top1,test_top3,test_mean_rank,test_avg_regret,test_avg_norm_regret
0,expanded_saved_h512_d2_e16_wd5e4,saved,512,16,2,0.05,mul_only,mse,36,0.018505,...,0.021825,0.048203,0.118466,0.086326,0.645658,0.316812,0.589417,4.198928,0.022763,0.045469
1,expanded_saved_h512_d2_e16_drop010,saved,512,16,2,0.10,mul_only,mse,30,0.023312,...,0.022876,0.049997,0.115619,0.083909,0.662486,0.358339,0.584059,4.152043,0.022200,0.044528
2,expanded_saved_h512_d2_e16_lr5e4,saved,512,16,2,0.05,mul_only,mse,34,0.026671,...,0.023013,0.048286,0.116906,0.086559,0.654928,0.290690,0.531815,4.434025,0.023385,0.046734
3,expanded_saved_h512_d2_e12,saved,512,12,2,0.05,mul_only,mse,13,0.045883,...,0.023032,0.048745,0.119445,0.089894,0.639781,0.221701,0.486269,4.837910,0.026475,0.052236
4,expanded_saved_h512_d2_e20,saved,512,20,2,0.05,mul_only,mse,50,0.014085,...,0.024476,0.051506,0.117138,0.085308,0.653557,0.313463,0.604823,4.383121,0.024184,0.048838


,run_name,train_utility,eval_objective,split,hidden,emb_dim,combine,top1,top3,mean_rank,avg_regret,avg_norm_regret
0,expanded_saved_h512_d2_e16_lr5e4,saved,contains_closest,test,512,16,mul_only,0.947756,0.947756,1.835901,0.052244,0.052244
1,expanded_saved_h512_d2_e16_drop010,saved,contains_closest,test,512,16,mul_only,0.945077,0.945077,1.878768,0.054923,0.054923
2,expanded_saved_h512_d2_e12,saved,contains_closest,test,512,12,mul_only,0.943737,0.943737,1.900201,0.056263,0.056263
3,expanded_saved_h512_d2_e16_wd5e4,saved,contains_closest,test,512,16,mul_only,0.941058,0.941058,1.943068,0.058942,0.058942
4,expanded_saved_h512_d2_e20,saved,contains_closest,test,512,20,mul_only,0.933691,0.933691,2.060951,0.066309,0.066309
5,expanded_saved_h512_d2_e16_drop010,saved,saved_rational,test,512,16,mul_only,0.358339,0.584059,4.152043,0.022200,0.044528
6,expanded_saved_h512_d2_e16_wd5e4,saved,saved_rational,test,512,16,mul_only,0.316812,0.589417,4.198928,0.022763,0.045469
7,expanded_saved_h512_d2_e16_lr5e4,saved,saved_rational,test,512,16,mul_only,0.290690,0.531815,4.434025,0.023385,0.046734
8,expanded_saved_h512_d2_e20,saved,saved_rational,test,512,20,mul_only,0.313463,0.604823,4.383121,0.024184,0.048838
9,expanded_saved_h512_d2_e12,saved,saved_rational,test,512,12,mul_only,0.221701,0.486269,4.837910,0.026475,0.052236


In [10]:
import numpy as np
import pandas as pd

from two_tower_training import (
    predict_scores,
    build_utility_labels,
)

def per_time_decisions(result, utility_name="saved", utility_kwargs=None, split_name="test"):
    """
    Returns one row per timestep:
    top1, top3, rank, regret, norm_regret, selected_value, best_value.
    """
    utility_kwargs = utility_kwargs or {}

    model = result["model"]
    prepared = result["prepared"]
    config = result["config"]
    device = result["device"]

    indices = prepared["split"][split_name]
    time_id = prepared["example_time_id"][indices]

    y_eval = build_utility_labels(
        examples_index=prepared["examples_index"],
        saved_y=prepared["saved_y_examples"],
        meta=prepared["meta"],
        utility_name=utility_name,
        utility_kwargs=utility_kwargs,
    ).astype(np.float32)[indices]

    scores = predict_scores(
        model,
        prepared,
        indices,
        config.batch_size * 2,
        device,
    )

    rows = []
    df = pd.DataFrame({
        "time_id": time_id,
        "y_true": y_eval,
        "score": scores,
    })

    for tid, g in df.groupby("time_id", sort=False):
        yt = g["y_true"].to_numpy(dtype=float)
        sc = g["score"].to_numpy(dtype=float)

        pred_idx = int(np.argmax(sc))
        selected_value = float(yt[pred_idx])
        best_value = float(np.max(yt))
        worst_value = float(np.min(yt))

        rank = int(np.sum(yt > selected_value + 1e-10)) + 1
        regret = best_value - selected_value
        denom = best_value - worst_value
        norm_regret = 0.0 if denom <= 1e-12 else regret / denom

        rows.append({
            "time_id": tid,
            "top1": float(rank == 1),
            "top3": float(rank <= 3),
            "rank": float(rank),
            "regret": regret,
            "norm_regret": norm_regret,
            "selected_value": selected_value,
            "best_value": best_value,
        })

    return pd.DataFrame(rows)


def paired_block_bootstrap(df_a, df_b, metric, higher_is_better=True, block_len=25, B=3000, seed=0):
    """
    Paired block bootstrap over consecutive time steps.
    Positive diff means A is better than B.
    """
    rng = np.random.default_rng(seed)

    a = df_a.sort_values("time_id")[metric].to_numpy()
    b = df_b.sort_values("time_id")[metric].to_numpy()

    assert len(a) == len(b), "A/B must have same number of timesteps."

    if higher_is_better:
        diff = a - b
    else:
        diff = b - a

    n = len(diff)
    starts = np.arange(0, max(1, n - block_len + 1))

    boot = []
    for _ in range(B):
        sampled = []
        while len(sampled) < n:
            s = int(rng.choice(starts))
            sampled.extend(diff[s:s + block_len])
        sampled = np.asarray(sampled[:n])
        boot.append(sampled.mean())

    boot = np.asarray(boot)

    return {
        "metric": metric,
        "A_mean": float(a.mean()),
        "B_mean": float(b.mean()),
        "A_minus_B_oriented": float(diff.mean()),
        "ci_low": float(np.quantile(boot, 0.025)),
        "ci_high": float(np.quantile(boot, 0.975)),
        "p_A_better": float(np.mean(boot > 0.0)),
    }


def compare_two_results_bootstrap(
    result_a,
    result_b,
    name_a,
    name_b,
    utility_name="saved",
    utility_kwargs=None,
    split_name="test",
    block_len=25,
):
    df_a = per_time_decisions(result_a, utility_name, utility_kwargs, split_name)
    df_b = per_time_decisions(result_b, utility_name, utility_kwargs, split_name)

    rows = []
    rows.append(paired_block_bootstrap(df_a, df_b, "top1", higher_is_better=True, block_len=block_len))
    rows.append(paired_block_bootstrap(df_a, df_b, "top3", higher_is_better=True, block_len=block_len))
    rows.append(paired_block_bootstrap(df_a, df_b, "rank", higher_is_better=False, block_len=block_len))
    rows.append(paired_block_bootstrap(df_a, df_b, "norm_regret", higher_is_better=False, block_len=block_len))
    rows.append(paired_block_bootstrap(df_a, df_b, "regret", higher_is_better=False, block_len=block_len))

    out = pd.DataFrame(rows)
    out.insert(0, "A", name_a)
    out.insert(1, "B", name_b)
    out.insert(2, "utility", utility_name)
    return out

# Build a name -> result dictionary from all result lists you still have.
all_results = []
for obj in [
    globals().get("expanded_results", []),
    globals().get("one_change_results", []),
    globals().get("e16_probe_results", []),
]:
    all_results.extend(obj)

result_by_name = {r["config"].run_name: r for r in all_results}

plain = result_by_name["expanded_saved_h512_d2_e16"]
drop = result_by_name["expanded_saved_h512_d2_e16_drop010"]
smooth = result_by_name["expanded_saved_h512_d2_e8_smoothl1"]

# Saved-rational comparison.
display(compare_two_results_bootstrap(
    plain,
    drop,
    "plain_e16",
    "e16_drop010",
    utility_name="saved",
    split_name="test",
    block_len=25,
))

# Contains-closest comparison.
display(compare_two_results_bootstrap(
    smooth,
    plain,
    "smoothl1_e8",
    "plain_e16",
    utility_name="closest_binary",
    split_name="test",
    block_len=25,
))

,A,B,utility,metric,A_mean,B_mean,A_minus_B_oriented,ci_low,ci_high,p_A_better
0,plain_e16,e16_drop010,saved,top1,0.358339,0.358339,0.000000,-0.024799,0.032150,0.586000
1,plain_e16,e16_drop010,saved,top3,0.606162,0.584059,0.022103,0.000000,0.050904,0.974000
2,plain_e16,e16_drop010,saved,rank,4.034159,4.152043,0.117883,-0.099866,0.352328,0.866000
3,plain_e16,e16_drop010,saved,norm_regret,0.042809,0.044528,0.001719,-0.002121,0.005784,0.818000
4,plain_e16,e16_drop010,saved,regret,0.021643,0.022200,0.000557,-0.001208,0.002376,0.758333


,A,B,utility,metric,A_mean,B_mean,A_minus_B_oriented,ci_low,ci_high,p_A_better
0,smoothl1_e8,plain_e16,closest_binary,top1,0.957803,0.945077,0.012726,-0.005358,0.033490,0.902333
1,smoothl1_e8,plain_e16,closest_binary,top3,0.957803,0.945077,0.012726,-0.005358,0.033490,0.902333
2,smoothl1_e8,plain_e16,closest_binary,rank,1.675151,1.878768,0.203617,-0.085733,0.535834,0.902333
3,smoothl1_e8,plain_e16,closest_binary,norm_regret,0.042197,0.054923,0.012726,-0.005358,0.033490,0.902333
4,smoothl1_e8,plain_e16,closest_binary,regret,0.042197,0.054923,0.012726,-0.005358,0.033490,0.902333
